In [ ]:


# ---------------------------------------------------------------------
# CONFIGURAÇÕES INICIAIS DAS ANÁLISES
# ---------------------------------------------------------------------
import sys
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Caminhos
SCRIPT_DIR = Path(__file__).resolve().parent
ANALYTICS_DIR = SCRIPT_DIR.parent

# Função de exportação
if str(ANALYTICS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYTICS_DIR))

from utils.export_utils import exportar_csv

PROJECT_ROOT = Path(__file__).resolve().parents[3]
OUTPUT_DIR = PROJECT_ROOT / "scripts" / "Analytics" / "outputs" / "gold_02"

# Spark
spark = (
    SparkSession.builder
    .appName("TechChallenge_Analytics")
    .master("local[*]")
    .getOrCreate()
)


# ---------------------------------------------------------------------
# PREPARAÇÃO DA GOLD 02 - Quais perfis profissionais são mais valorizados pelo mercado?
# ---------------------------------------------------------------------
caminho_gold_02 = PROJECT_ROOT / "Gold" / "perguntas_negocio" / "gold_02_perfis_valorizados"

arquivos_gold_02 = [str(arquivo) for arquivo in caminho_gold_02.glob("part-*.csv")]

if not arquivos_gold_02:
    raise FileNotFoundError(f"Nenhum arquivo part-*.csv encontrado em: {caminho_gold_02}")

# Carregar Gold 02
df_perfis = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(arquivos_gold_02)
)


# ---------------------------------------------------------------------
# TRATAMENTO DA BASE
# ---------------------------------------------------------------------
# Remover faixa salarial inconsistente
"""
A faixa identificada como inconsistente é removida antes das análises históricas para não distorcer a distribuição salarial nem os percentis calculados nas três edições.
"""
df_perfis_tratado = (
    df_perfis.filter(
        (F.col("valor") != "de R$ 101/mês a R$ 2.000/mês") | F.col("valor").isNull()
    )
)

# Harmonizar Engenharia e Arquitetura de Dados
"""
As variações de nomenclatura de Engenharia e Arquitetura de Dados são reunidas em uma única categoria para evitar fragmentação artificial de um mesmo perfil profissional ao longo do histórico.
"""
df_perfis_tratado = (
    df_perfis_tratado.withColumn(
        "cargo_harmonizado",
        F.when(
            F.col("cargo_atual").isin(
                "Engenheiro de Dados/Arquiteto de Dados/Data Engineer/Data Architect",
                "Engenheiro de Dados/Data Engineer/Data Architect",
                "Arquiteto de Dados/Data Architect"
            ),
            "Engenharia e Arquitetura de Dados"
        ).otherwise(F.col("cargo_atual"))
    )
)

# Consolidar contagens após harmonização
df_perfis_consolidado = (
    df_perfis_tratado
    .groupBy("edicao", "cargo_harmonizado", "nivel", "valor")
    .agg(F.sum("contagem").alias("contagem"))
)

# Recalcular total de respondentes
"""
Após os tratamentos, o total de respondentes é recalculado por edição, cargo harmonizado e nível. Esse total tratado passa a ser o denominador das proporções e da distribuição acumulada usada nos percentis.
"""
janela_perfil = Window.partitionBy("edicao", "cargo_harmonizado", "nivel")

df_perfis_consolidado = (
    df_perfis_consolidado
    .withColumn("total_respondentes", F.sum("contagem").over(janela_perfil))
    .withColumn(
        "pct_na_dimensao",
        F.round((F.col("contagem") / F.col("total_respondentes")) * 100, 2)
    )
)


# ---------------------------------------------------------------------
# ORDENAR AS FAIXAS SALARIAIS
# ---------------------------------------------------------------------
"""
As faixas salariais são convertidas em uma ordem numérica crescente porque a análise depende de comparar posições relativas entre intervalos, e não valores salariais contínuos.
"""
df_perfis_ordenado = (
    df_perfis_consolidado.withColumn(
        "ordem_faixa_salarial",
        F.when(F.col("valor") == "Menos de R$ 1.000/mês", 1)
        .when(F.col("valor") == "de R$ 1.001/mês a R$ 2.000/mês", 2)
        .when(F.col("valor") == "de R$ 2.001/mês a R$ 3.000/mês", 3)
        .when(F.col("valor") == "de R$ 3.001/mês a R$ 4.000/mês", 4)
        .when(F.col("valor") == "de R$ 4.001/mês a R$ 6.000/mês", 5)
        .when(F.col("valor") == "de R$ 6.001/mês a R$ 8.000/mês", 6)
        .when(F.col("valor") == "de R$ 8.001/mês a R$ 12.000/mês", 7)
        .when(F.col("valor") == "de R$ 12.001/mês a R$ 16.000/mês", 8)
        .when(F.col("valor") == "de R$ 16.001/mês a R$ 20.000/mês", 9)
        .when(F.col("valor") == "de R$ 20.001/mês a R$ 25.000/mês", 10)
        .when(F.col("valor") == "de R$ 25.001/mês a R$ 30.000/mês", 11)
        .when(F.col("valor") == "de R$ 30.001/mês a R$ 40.000/mês", 12)
        .when(F.col("valor") == "Acima de R$ 40.001/mês", 13)
    )
)


# ---------------------------------------------------------------------
# SELECIONAR PERFIS PARA O HISTÓRICO
# ---------------------------------------------------------------------
print("\n" + "=" * 100)
print("3. PERFIS SELECIONADOS PARA ANÁLISE HISTÓRICA")
print("=" * 100)

# Perfis que apareceram entre os destaques da edição atual
"""
O histórico não é aberto para todos os cargos da base. Ele foca nos perfis que apareceram entre os destaques da edição mais recente, para aprofundar a leitura dos perfis mais valorizados no cenário atual.
"""
perfis_historico = [
    "Cientista de Dados/Data Scientist",
    "Engenheiro de Machine Learning/ML Engineer/AI Engineer",
    "Engenharia e Arquitetura de Dados",
    "Analytics Engineer",
    "Desenvolvedor/ Engenheiro de Software/ Analista de Sistemas"
]

# Níveis comparáveis nas três edições
"""
O recorte considera apenas Júnior, Pleno e Sênior, que são os níveis com comparabilidade mais direta entre as três edições analisadas.
"""
niveis_historico = ["Júnior", "Pleno", "Sênior"]

df_historico_perfis = (
    df_perfis_ordenado.filter(
        F.col("cargo_harmonizado").isin(perfis_historico)
        & F.col("nivel").isin(niveis_historico)
    )
)


# ---------------------------------------------------------------------
# VALIDAR TAMANHO DAS AMOSTRAS NO HISTÓRICO
# ---------------------------------------------------------------------
print("\n" + "=" * 100)
print("4. COMPARABILIDADE DAS AMOSTRAS HISTÓRICAS")
print("=" * 100)

amostra_historico = (
    df_historico_perfis
    .select("edicao", "cargo_harmonizado", "nivel", "total_respondentes")
    .distinct()
)

resumo_amostra_historico = (
    amostra_historico
    .groupBy("cargo_harmonizado", "nivel")
    .agg(
        F.countDistinct("edicao").alias("qtd_edicoes"),
        F.min("total_respondentes").alias("menor_amostra"),
        F.max("total_respondentes").alias("maior_amostra")
    )
    .orderBy("cargo_harmonizado", "nivel")
)

resumo_amostra_historico.show(100, truncate=False)
"""
A validação de amostra serve para separar presença histórica de comparabilidade analítica. Pelos outputs, alguns perfis existem nas três edições, mas não sustentam comparação robusta porque a menor amostra fica abaixo do corte adotado.
"""


# ---------------------------------------------------------------------
# SELECIONAR PERFIS ELEGÍVEIS PARA O HISTÓRICO
# ---------------------------------------------------------------------
print("\n" + "=" * 100)
print("5. PERFIS ELEGÍVEIS PARA COMPARAÇÃO HISTÓRICA")
print("=" * 100)

# O perfil precisa existir nas 3 edições e manter pelo menos 20 respondentes em cada uma delas.
"""
O critério combina continuidade histórica e tamanho mínimo de amostra. Isso evita comparar perfis que aparecem no tempo, mas com volume insuficiente em alguma edição.
"""
perfis_elegiveis_historico = (
    resumo_amostra_historico
    .filter((F.col("qtd_edicoes") == 3) & (F.col("menor_amostra") >= 20))
    .orderBy("cargo_harmonizado", "nivel")
)

perfis_elegiveis_historico.show(100, truncate=False)
print("Quantidade de perfis elegíveis:", perfis_elegiveis_historico.count())
"""
Na execução do notebook, 12 combinações cargo + nível permaneceram elegíveis. Analytics Engineer Júnior e Engenheiro de Machine Learning/ML Engineer/AI Engineer nos níveis Júnior e Pleno ficaram de fora porque a menor amostra observada foi inferior a 20 respondentes.
"""


# ---------------------------------------------------------------------
# PREPARAR BASE HISTÓRICA ELEGÍVEL
# ---------------------------------------------------------------------
chaves_perfis_historico = perfis_elegiveis_historico.select("cargo_harmonizado", "nivel")

df_historico_elegivel = (
    df_historico_perfis.join(
        chaves_perfis_historico,
        on=["cargo_harmonizado", "nivel"],
        how="inner"
    )
)


# ---------------------------------------------------------------------
# CALCULAR DISTRIBUIÇÃO ACUMULADA POR EDIÇÃO
"""
A base histórica elegível é a que efetivamente sustenta o cálculo dos percentis. O join interno garante que somente os perfis aprovados na checagem de comparabilidade avancem para as etapas seguintes.
"""
# ---------------------------------------------------------------------
"""
A distribuição é acumulada separadamente por edição, cargo e nível para que o P50 e o P75 reflitam a posição salarial de cada perfil em cada momento do histórico.
"""
janela_acumulada_historico = (
    Window
    .partitionBy("edicao", "cargo_harmonizado", "nivel")
    .orderBy("ordem_faixa_salarial")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_historico_acumulado = (
    df_historico_elegivel
    .withColumn("contagem_acumulada", F.sum("contagem").over(janela_acumulada_historico))
    .withColumn(
        "pct_acumulado",
        F.round((F.col("contagem_acumulada") / F.col("total_respondentes")) * 100, 2)
    )
)


# ---------------------------------------------------------------------
# CALCULAR P50 HISTÓRICO
"""
O P50 representa a primeira faixa em que a distribuição acumulada atinge 50% dos respondentes. Como a remuneração está organizada em intervalos, a mediana é lida por faixa e não por valor exato.
"""
# ---------------------------------------------------------------------
janela_percentil_historico = (
    Window
    .partitionBy("edicao", "cargo_harmonizado", "nivel")
    .orderBy("ordem_faixa_salarial")
)

historico_p50 = (
    df_historico_acumulado
    .filter(F.col("pct_acumulado") >= 50)
    .withColumn("ordem_p50", F.row_number().over(janela_percentil_historico))
    .filter(F.col("ordem_p50") == 1)
    .select(
        "edicao", "cargo_harmonizado", "nivel", "total_respondentes",
        F.col("valor").alias("faixa_p50"),
        F.col("ordem_faixa_salarial").alias("ordem_faixa_p50")
    )
)


# ---------------------------------------------------------------------
# CALCULAR P75 HISTÓRICO
"""
O P75 complementa a mediana ao capturar a parte superior da distribuição. Ele ajuda a identificar movimentos salariais que não aparecem no P50, mas surgem entre os 25% com remuneração mais alta.
"""
# ---------------------------------------------------------------------
historico_p75 = (
    df_historico_acumulado
    .filter(F.col("pct_acumulado") >= 75)
    .withColumn("ordem_p75", F.row_number().over(janela_percentil_historico))
    .filter(F.col("ordem_p75") == 1)
    .select(
        "edicao", "cargo_harmonizado", "nivel",
        F.col("valor").alias("faixa_p75"),
        F.col("ordem_faixa_salarial").alias("ordem_faixa_p75")
    )
)


# ---------------------------------------------------------------------
# CONSOLIDAR P50 E P75 HISTÓRICOS
# ---------------------------------------------------------------------
historico_p50_p75 = (
    historico_p50
    .join(
        historico_p75,
        on=["edicao", "cargo_harmonizado", "nivel"],
        how="inner"
    )
    .orderBy("cargo_harmonizado", "nivel", "edicao")
)

print("\n" + "=" * 100)
print("10. P50 E P75 HISTÓRICOS")
print("=" * 100)

historico_p50_p75.show(100, truncate=False)
"""
Os resultados mostram que parte dos perfis permaneceu estável nas três edições, enquanto outros tiveram avanço em P50 ou P75. Esse quadro orienta a etapa seguinte, focada em medir a variação entre o início e o fim da série.
"""


# ---------------------------------------------------------------------
# COMPARAR PRIMEIRA E ÚLTIMA EDIÇÃO
"""
A comparação direta entre 2023-2024 e 2025-2026 resume a variação líquida do período analisado. As faixas são comparadas por sua ordem numérica para medir quantos intervalos cada perfil avançou, recuou ou manteve.
"""
# ---------------------------------------------------------------------
print("\n" + "=" * 100)
print("11. VARIAÇÃO DAS FAIXAS SALARIAIS - 2023-2024 x 2025-2026")
print("=" * 100)

historico_inicio = (
    historico_p50_p75
    .filter(F.col("edicao") == "2023-2024")
    .select(
        "cargo_harmonizado", "nivel",
        F.col("faixa_p50").alias("p50_2023_2024"),
        F.col("ordem_faixa_p50").alias("ordem_p50_2023_2024"),
        F.col("faixa_p75").alias("p75_2023_2024"),
        F.col("ordem_faixa_p75").alias("ordem_p75_2023_2024")
    )
)

historico_fim = (
    historico_p50_p75
    .filter(F.col("edicao") == "2025-2026")
    .select(
        "cargo_harmonizado", "nivel",
        F.col("faixa_p50").alias("p50_2025_2026"),
        F.col("ordem_faixa_p50").alias("ordem_p50_2025_2026"),
        F.col("faixa_p75").alias("p75_2025_2026"),
        F.col("ordem_faixa_p75").alias("ordem_p75_2025_2026")
    )
)

evolucao_perfis = (
    historico_inicio
    .join(
        historico_fim,
        on=["cargo_harmonizado", "nivel"],
        how="inner"
    )
    .withColumn(
        "variacao_faixas_p50",
        F.col("ordem_p50_2025_2026") - F.col("ordem_p50_2023_2024")
    )
    .withColumn(
        "variacao_faixas_p75",
        F.col("ordem_p75_2025_2026") - F.col("ordem_p75_2023_2024")
    )
    .orderBy(
        F.desc("variacao_faixas_p50"),
        F.desc("variacao_faixas_p75"),
        "cargo_harmonizado",
        "nivel"
    )
)

evolucao_perfis.show(100, truncate=False)
"""
Nos outputs, a maior variação de P75 aparece em Engenheiro de Machine Learning/ML Engineer/AI Engineer Sênior, com avanço de duas faixas. Também se destacam os avanços de Analytics Engineer Pleno no P50 e de Desenvolvedor/ Engenheiro de Software/ Analista de Sistemas Júnior em P50 e P75.
"""


# ---------------------------------------------------------------------
# IDENTIFICAR MOVIMENTOS SALARIAIS RELEVANTES
"""
Perfis sem mudança em P50 e P75 são removidos deste recorte porque a etapa busca destacar apenas movimentos históricos relevantes para o storytelling executivo.
"""
# ---------------------------------------------------------------------
print("\n" + "=" * 100)
print("12. MOVIMENTOS SALARIAIS HISTÓRICOS RELEVANTES")
print("=" * 100)

movimentos_historicos = (
    evolucao_perfis
    .filter(
        (F.col("variacao_faixas_p50") != 0)
        | (F.col("variacao_faixas_p75") != 0)
    )
    .select(
        "cargo_harmonizado", "nivel", "p50_2023_2024", "p50_2025_2026",
        "variacao_faixas_p50", "p75_2023_2024", "p75_2025_2026",
        "variacao_faixas_p75"
    )
    .orderBy(
        F.desc("variacao_faixas_p75"),
        F.desc("variacao_faixas_p50"),
        "cargo_harmonizado",
        "nivel"
    )
)

movimentos_historicos.show(100, truncate=False)
print("Quantidade de perfis com mudança:", movimentos_historicos.count())
"""
A filtragem final identificou 5 combinações com alguma mudança salarial entre a primeira e a última edição. Esse subconjunto é o que segue para a visualização histórica detalhada.
"""


# ---------------------------------------------------------------------
# PREPARAR RESULTADO FINAL PARA VISUALIZAÇÃO
# ---------------------------------------------------------------------
# Combinações que apresentaram mudança
perfis_com_movimento = movimentos_historicos.select("cargo_harmonizado", "nivel").distinct()

# Preservar as três edições para os gráficos
"""
Mesmo quando a mudança é medida entre a primeira e a última edição, as três edições são mantidas no resultado final para permitir visualizar a trajetória intermediária dos perfis que se moveram.
"""
perfis_evolucao_historica = (
    historico_p50_p75
    .join(
        perfis_com_movimento,
        on=["cargo_harmonizado", "nivel"],
        how="inner"
    )
    .select(
        "edicao", "cargo_harmonizado", "nivel", "total_respondentes",
        "faixa_p50", "ordem_faixa_p50", "faixa_p75", "ordem_faixa_p75"
    )
    .orderBy("cargo_harmonizado", "nivel", "edicao")
)

print("\n" + "=" * 100)
print("13. EVOLUÇÃO DOS PERFIS COM MUDANÇA SALARIAL")
print("=" * 100)

perfis_evolucao_historica.show(100, truncate=False)


# ---------------------------------------------------------------------
# EXPORTAÇÃO PARA VISUALIZAÇÃO
# ---------------------------------------------------------------------
exportar_csv(perfis_evolucao_historica, OUTPUT_DIR, "perfis_evolucao_historica.csv")